# Subject–Object Relations

Factory task over 35 LRE subject→object relations (`subject → object → raw_output`, `subject → raw_input`). Select a relation with `SubjectObjectRelationsConfig(relation=<name>)`. CPU-only: this notebook exercises the *task* (causal model, samples, token positions, counterfactuals), never a language model.

In [ ]:
from causalab.tasks.subject_object_relations import (
    SubjectObjectRelationsConfig,
    create_causal_model,
    relation_names,
    COUNTERFACTUAL_GENERATORS,
)
from causalab.tasks.subject_object_relations import counterfactuals as cf

print(len(relation_names()), "relations available")
print("first few:", relation_names()[:6])

RELATION = "word_first_letter"  # a green relation; try 'name_gender', 'substance_phase'
config = SubjectObjectRelationsConfig(relation=RELATION)
CAUSAL_MODEL = create_causal_model(config)

## Causal Model Variables

In [ ]:
print("variables:", CAUSAL_MODEL.variables)
print("n subjects:", len(CAUSAL_MODEL.values["subject"]))
print("n objects (answer space):", len(CAUSAL_MODEL.values["object"]))
print("parents:", {v: CAUSAL_MODEL.parents[v] for v in CAUSAL_MODEL.variables})
print("group / category:", config.group, "/", config.category)
print("templates:", config.templates)

## Sample Generation

In [ ]:
for _ in range(5):
    s = CAUSAL_MODEL.sample_input()
    print(repr(s["raw_input"]), "->", repr(s["raw_output"]))

## Token Positions

In [ ]:
# token_positions.create_token_positions(pipeline, template=...) needs a live
# pipeline to materialise indices, so it is not run in this model-free notebook.
# Declarative specs (see token_positions.py):
#   last_token : final prompt token (index -1)
#   subject    : last token of the {subject} span
print("template used for positions:", config.templates[0])

## Counterfactual Generation (object flip)

In [ ]:
examples = cf.generate_dataset(CAUSAL_MODEL, n=4, seed=0)
for ex in examples:
    base = ex["input"]
    cfi = ex["counterfactual_inputs"][0]
    print(f"base:  {base['raw_input']!r} -> {base['object']!r}")
    print(f"cf:    {cfi['raw_input']!r} -> {cfi['object']!r}")
    print(f"       object flips: {base['object'] != cfi['object']}\n")

print("registered generators:", list(COUNTERFACTUAL_GENERATORS))